# Parameter Freedom Analysis for SBML Models

**Mathematical approach to identify truly free parameters in Systems Biology models**

This notebook implements rigorous parameter freedom analysis to replace crude name-based filtering approaches. It uses:

1. **Structural Analysis**: Examines SBML rules, assignments, and stoichiometry
2. **Sensitivity Analysis**: Tests which parameters actually affect model dynamics
3. **Conservation Detection**: Identifies parameters constrained by conservation laws

**Target**: Identify the ~54 truly free parameters in Chen 2004 budding yeast cell cycle model instead of using crude name-based heuristics that may include hundreds of parameters.

In [1]:
# === IMPORTS ===
import tellurium as te
import numpy as np
import matplotlib.pyplot as plt
import libsbml
import os
import platform
from typing import Dict, Set, Tuple, List
import warnings
import roadrunner

# Silence warnings and RoadRunner logs for cleaner output
warnings.filterwarnings('ignore')
roadrunner.Logger.setLevel(roadrunner.Logger.LOG_CRITICAL)

print("🧬 Parameter Freedom Analysis for SBML Models")
print("📊 Mathematical approach to identify truly free parameters")
print("="*60)

🧬 Parameter Freedom Analysis for SBML Models
📊 Mathematical approach to identify truly free parameters


In [2]:
# === MODEL PATH DETECTION ===
def get_model_path():
    """Get the correct path to the Chen 2004 SBML model file."""
    possible_paths = [
        "/home/gijs/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml",
        "/home/b/bartholomeus/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml",   
        "/Users/gijsbartholomeus/Documents/STUDIE/OxfordEvolution/code/Yeast/Chen/chen_model.xml",
        "chen_model.xml",
        "Chen/chen_model.xml",
        "../Chen/chen_model.xml"
    ]
    
    for path in possible_paths:
        if os.path.exists(path):
            print(f"Found model at: {path}")
            return path
    
    raise FileNotFoundError(
        f"Could not find chen_model.xml in any expected location.\n"
        f"Current directory: {os.getcwd()}\n"
        f"Platform: {platform.system()}"
    )

# Load the Chen 2004 model
MODEL_PATH = get_model_path()
print(f"✓ Loading Chen 2004 budding yeast cell cycle model")

Found model at: /home/gijs/Documents/OxfordEvolution/Yeast/Chen/chen_model.xml
✓ Loading Chen 2004 budding yeast cell cycle model


In [3]:
# === PARAMETER FREEDOM ANALYSIS CLASSES ===

class ParameterAnalyzer:
    """Rigorous parameter freedom analysis for SBML models"""
    
    def __init__(self, model_path: str):
        self.model_path = model_path
        self.rr = te.loadSBMLModel(model_path)
        self.sbml_doc = libsbml.readSBML(model_path)
        self.model = self.sbml_doc.getModel()
        
    def get_free_parameters(self) -> Dict[str, float]:
        """
        Identify truly free parameters using SBML structure analysis.
        
        Returns:
            Dictionary of {param_name: current_value} for free parameters only
        """
        print("   🔍 Analyzing SBML structure...")
        
        # Start with all global parameters
        all_params = {p.getId(): p.getValue() 
                      for p in self.model.getListOfParameters()}
        print(f"   📊 Found {len(all_params)} total parameters")
        
        # Remove parameters defined by rules
        rule_params = self._get_rule_defined_parameters()
        print(f"   📏 Rule-defined parameters: {len(rule_params)}")
        
        # Remove parameters with initial assignments
        initial_assignment_params = self._get_initial_assignment_parameters()
        print(f"   🎯 Initial assignment parameters: {len(initial_assignment_params)}")
        
        # Remove stoichiometric constants if they're exposed as parameters
        stoich_params = self._get_stoichiometric_parameters()
        print(f"   ⚖️  Stoichiometric parameters: {len(stoich_params)}")
        
        # Get conservation law dependent parameters
        conservation_params = self._get_conservation_dependent_parameters()
        print(f"   🔄 Conservation-dependent parameters: {len(conservation_params)}")
        
        # Combine all non-free parameters
        non_free = (rule_params | initial_assignment_params | 
                    stoich_params | conservation_params)
        
        free_params = {k: v for k, v in all_params.items() 
                       if k not in non_free}
        
        print(f"   ✅ Structurally free parameters: {len(free_params)}")
        return free_params
    
    def _get_rule_defined_parameters(self) -> Set[str]:
        """Find parameters defined by assignment or rate rules"""
        rule_params = set()
        
        for rule in self.model.getListOfRules():
            if isinstance(rule, libsbml.AssignmentRule):
                rule_params.add(rule.getVariable())
            elif isinstance(rule, libsbml.RateRule):
                rule_params.add(rule.getVariable())
                
        return rule_params
    
    def _get_initial_assignment_parameters(self) -> Set[str]:
        """Find parameters with initial assignments (SBML L3 feature)"""
        init_params = set()
        
        for init_assign in self.model.getListOfInitialAssignments():
            init_params.add(init_assign.getSymbol())
            
        return init_params
    
    def _get_stoichiometric_parameters(self) -> Set[str]:
        """
        Identify parameters that are actually stoichiometric coefficients.
        These shouldn't be varied in typical bifurcation analysis.
        """
        stoich_params = set()
        
        for reaction in self.model.getListOfReactions():
            # Check reactants
            for reactant in reaction.getListOfReactants():
                if reactant.isSetStoichiometry() and reactant.isSetId():
                    stoich_params.add(reactant.getId())
                    
            # Check products
            for product in reaction.getListOfProducts():
                if product.isSetStoichiometry() and product.isSetId():
                    stoich_params.add(product.getId())
                    
        return stoich_params
    
    def _get_conservation_dependent_parameters(self) -> Set[str]:
        """
        Detect parameters constrained by conservation laws.
        For models with rule-defined species, we skip automatic conservation detection.
        """
        conservation_params = set()
        
        try:
            # Create a fresh RoadRunner instance with conservation analysis
            import roadrunner
            options = roadrunner.LoadSBMLOptions()
            options.conservedMoieties = True
            
            temp_rr = roadrunner.RoadRunner()
            temp_rr.load(self.sbml_doc.toXMLString(), options)
            
            # Get dependent species
            dependent_species = temp_rr.getDependentFloatingSpeciesIds()
            
            # Look for total/conservation parameters
            all_param_ids = [p.getId() for p in self.model.getListOfParameters()]
            for species in dependent_species:
                possible_params = [
                    f"{species}_total", f"{species}_0", f"total_{species}", f"{species}T",
                    f"{species.upper()}_total", f"{species.upper()}T"
                ]
                for param_name in possible_params:
                    if param_name in all_param_ids:
                        conservation_params.add(param_name)
                        
        except Exception:
            print("     Note: Conservation analysis skipped (rule-defined species)")
            
        return conservation_params


class ParameterSensitivityAnalyzer:
    """
    Empirical sensitivity analysis to validate structural analysis.
    Identifies parameters that actually affect model dynamics.
    """
    
    def __init__(self, rr):
        self.rr = rr
        
    def sensitivity_analysis(self, 
                            param_names: List[str],
                            perturbation: float = 0.01,
                            observable: str = 'CLB2',
                            simulation_time: float = 400,
                            show_progress: bool = True) -> Dict[str, float]:
        """
        Perform local sensitivity analysis.
        
        Args:
            param_names: Parameters to test
            perturbation: Relative perturbation size (default 1%)
            observable: Species to measure (default CLB2)
            simulation_time: Simulation duration
            show_progress: Show progress during analysis
            
        Returns:
            Dictionary of {param: sensitivity_score}
        """
        print(f"   🧪 Testing sensitivity of {len(param_names)} parameters...")
        
        sensitivities = {}
        
        # Set minimal selections for performance
        original_selections = self.rr.selections
        self.rr.selections = ["time", observable]
        
        # Baseline simulation
        try:
            self.rr.reset()
            baseline = self.rr.simulate(0, simulation_time, 1000)
            baseline_series = baseline[observable]
            baseline_metric = self._compute_metric(baseline_series)
        except Exception as e:
            print(f"   ❌ Baseline simulation failed: {e}")
            self.rr.selections = original_selections
            return {}
        
        # Test each parameter
        failed_params = []
        progress_interval = max(1, len(param_names) // 20)
        
        for i, param in enumerate(param_names):
            if show_progress and i % progress_interval == 0:
                progress = (i / len(param_names)) * 100
                print(f"   📈 Progress: {progress:.0f}% ({i}/{len(param_names)})")
            
            try:
                original_value = self.rr[param]
                
                # Skip if parameter is zero (can't do relative perturbation)
                if abs(original_value) < 1e-12:
                    sensitivities[param] = 0.0
                    continue
                
                # Perturb parameter
                self.rr[param] = original_value * (1 + perturbation)
                
                # Simulate
                self.rr.reset()
                perturbed = self.rr.simulate(0, simulation_time, 1000)
                perturbed_series = perturbed[observable]
                perturbed_metric = self._compute_metric(perturbed_series)
                
                # Compute normalized sensitivity
                relative_change = abs(perturbed_metric - baseline_metric) / abs(baseline_metric + 1e-12)
                sensitivity = relative_change / perturbation
                sensitivities[param] = sensitivity
                
                # Restore original value
                self.rr[param] = original_value
                
            except Exception as e:
                failed_params.append(param)
                sensitivities[param] = 0.0
                try:
                    self.rr[param] = original_value
                except:
                    pass
        
        # Report failed parameters
        if failed_params:
            print(f"   ⚠️  Failed sensitivity tests: {len(failed_params)} parameters")
            if len(failed_params) <= 5:
                print(f"     {', '.join(failed_params)}")
        
        # Restore original selections
        self.rr.selections = original_selections
        print(f"   ✅ Sensitivity analysis complete")
        
        return sensitivities
    
    def _compute_metric(self, timeseries: np.ndarray) -> float:
        """
        Compute summary metric for timeseries.
        Uses multiple metrics for robustness.
        """
        if len(timeseries) == 0:
            return 0.0
            
        # Use multiple metrics for robustness
        mean_val = np.mean(timeseries)
        std_val = np.std(timeseries)
        max_val = np.max(timeseries)
        
        # Combined metric (weighted sum to avoid cancellation)
        return abs(mean_val) + std_val + abs(max_val)

print("✅ Parameter freedom analysis classes defined!")

✅ Parameter freedom analysis classes defined!


In [4]:
# === ANALYSIS FUNCTIONS ===

def analyze_parameter_freedom(model_path: str, 
                             sensitivity_threshold: float = 1e-4,
                             simulation_time: float = 400,
                             observable: str = 'CLB2') -> Tuple[Dict[str, float], Dict[str, float], Dict]:
    """
    Complete parameter freedom analysis of SBML model.
    
    Args:
        model_path: Path to SBML file
        sensitivity_threshold: Minimum sensitivity to consider parameter "active"
        simulation_time: Duration for sensitivity testing
        observable: Species to monitor for sensitivity
        
    Returns:
        (active_parameters, all_free_parameters, sensitivity_scores, analysis_info)
    """
    print("🔬 PARAMETER FREEDOM ANALYSIS")
    print("="*50)
    
    # Step 1: Structural analysis
    print("🧬 Step 1: Structural Analysis")
    analyzer = ParameterAnalyzer(model_path)
    free_params = analyzer.get_free_parameters()
    
    # Step 2: Sensitivity analysis
    print(f"\n🎯 Step 2: Sensitivity Analysis (threshold: {sensitivity_threshold})")
    sens_analyzer = ParameterSensitivityAnalyzer(analyzer.rr)
    sensitivities = sens_analyzer.sensitivity_analysis(
        list(free_params.keys()),
        simulation_time=simulation_time,
        observable=observable
    )
    
    # Step 3: Filter by sensitivity
    print("\n📊 Step 3: Filtering by Sensitivity")
    active_params = {k: v for k, v in free_params.items() 
                     if sensitivities.get(k, 0) > sensitivity_threshold}
    
    # Analysis summary
    analysis_info = {
        'total_parameters': len([p.getId() for p in analyzer.model.getListOfParameters()]),
        'structurally_free': len(free_params),
        'sensitivity_active': len(active_params),
        'threshold': sensitivity_threshold,
        'simulation_time': simulation_time,
        'observable': observable
    }
    
    print(f"\n📈 RESULTS SUMMARY:")
    print(f"   Total parameters in model: {analysis_info['total_parameters']}")
    print(f"   Structurally free: {analysis_info['structurally_free']}")
    print(f"   Sensitivity-active: {analysis_info['sensitivity_active']}")
    print(f"   Reduction factor: {analysis_info['total_parameters']/analysis_info['sensitivity_active']:.1f}x")
    
    # Show insensitive parameters
    insensitive = set(free_params.keys()) - set(active_params.keys())
    if insensitive:
        print(f"\n   Structurally free but insensitive parameters ({len(insensitive)}):")
        if len(insensitive) <= 10:
            for param in sorted(insensitive):
                sens_val = sensitivities.get(param, 0)
                print(f"     {param}: {sens_val:.2e}")
        else:
            print(f"     (Too many to display: {len(insensitive)} parameters)")
            # Show just the first few
            for param in sorted(list(insensitive)[:5]):
                sens_val = sensitivities.get(param, 0)
                print(f"     {param}: {sens_val:.2e}")
            print(f"     ... and {len(insensitive) - 5} more")
    
    return active_params, free_params, sensitivities, analysis_info


def compare_with_crude_filtering(rr, active_params: Dict[str, float]) -> Dict:
    """
    Compare rigorous analysis with crude name-based filtering.
    
    Args:
        rr: RoadRunner instance
        active_params: Result from rigorous analysis
        
    Returns:
        Comparison dictionary
    """
    print("\n🆚 COMPARISON WITH CRUDE NAME-BASED FILTERING")
    print("="*50)
    
    # Implement crude filtering (legacy approach)
    crude_params = []
    
    for pid in rr.getGlobalParameterIds():
        value = rr.getValue(pid)
        param_lower = pid.lower()
        
        # Original crude exclusion rules
        excluded = (
            (param_lower.endswith('t') and value in [0.0, 1.0]) or
            (param_lower.startswith('d') and param_lower.endswith('n')) or
            ('flag' in param_lower) or
            ('switch' in param_lower) or
            (value == 0.0) or
            (pid in ['cell']) or
            ('total' in param_lower and value in [0.0, 1.0])
        )
        
        if not excluded:
            crude_params.append(pid)
    
    # Compare results
    rigorous_set = set(active_params.keys())
    crude_set = set(crude_params)
    
    only_rigorous = rigorous_set - crude_set
    only_crude = crude_set - rigorous_set
    common = rigorous_set & crude_set
    
    comparison = {
        'rigorous_count': len(rigorous_set),
        'crude_count': len(crude_set),
        'common_count': len(common),
        'only_rigorous': only_rigorous,
        'only_crude': only_crude,
        'reduction_factor': len(crude_set) / len(rigorous_set) if len(rigorous_set) > 0 else 0
    }
    
    print(f"📊 Comparison Results:")
    print(f"   Rigorous analysis: {comparison['rigorous_count']} parameters")
    print(f"   Crude filtering: {comparison['crude_count']} parameters")
    print(f"   Common parameters: {comparison['common_count']}")
    print(f"   Improvement: {comparison['reduction_factor']:.1f}x fewer parameters")
    
    if only_crude:
        print(f"\n   Parameters incorrectly included by crude method ({len(only_crude)}):")
        if len(only_crude) <= 10:
            for param in sorted(only_crude):
                print(f"     {param}")
        else:
            for param in sorted(list(only_crude)[:5]):
                print(f"     {param}")
            print(f"     ... and {len(only_crude) - 5} more")
    
    if only_rigorous:
        print(f"\n   Parameters correctly identified by rigorous method ({len(only_rigorous)}):")
        if len(only_rigorous) <= 5:
            for param in sorted(only_rigorous):
                print(f"     {param}")
        else:
            for param in sorted(list(only_rigorous)[:3]):
                print(f"     {param}")
            print(f"     ... and {len(only_rigorous) - 3} more")
    
    return comparison

print("✅ Analysis functions defined!")

✅ Analysis functions defined!


In [5]:
# === PERFORM ANALYSIS ON CHEN 2004 MODEL ===

print("🧬 ANALYZING CHEN 2004 BUDDING YEAST CELL CYCLE MODEL")
print("🎯 Goal: Identify truly free parameters for complexity analysis")
print("📚 Expected from SloppyCell literature: ~54 parameters")
print("\n" + "="*70)

# Run the complete analysis
active_params, all_free_params, sensitivities, info = analyze_parameter_freedom(
    MODEL_PATH,
    sensitivity_threshold=1e-4,
    simulation_time=400,
    observable='CLB2'
)

print(f"\n🎯 FINAL RESULT: {len(active_params)} mathematically sound free parameters identified")
print(f"📈 Literature target: ~54 parameters")
print(f"📊 Gap: {len(active_params) - 54} parameters above expected")

🧬 ANALYZING CHEN 2004 BUDDING YEAST CELL CYCLE MODEL
🎯 Goal: Identify truly free parameters for complexity analysis
📚 Expected from SloppyCell literature: ~54 parameters

🔬 PARAMETER FREEDOM ANALYSIS
🧬 Step 1: Structural Analysis
   🔍 Analyzing SBML structure...
   📊 Found 163 total parameters
   📏 Rule-defined parameters: 35
   🎯 Initial assignment parameters: 0
   ⚖️  Stoichiometric parameters: 0
     Note: Conservation analysis skipped (rule-defined species)
   🔄 Conservation-dependent parameters: 0
   ✅ Structurally free parameters: 143

🎯 Step 2: Sensitivity Analysis (threshold: 0.0001)
   🧪 Testing sensitivity of 143 parameters...
   📈 Progress: 0% (0/143)
   📈 Progress: 5% (7/143)
   📈 Progress: 10% (14/143)
   📈 Progress: 15% (21/143)
   📈 Progress: 20% (28/143)
   📈 Progress: 24% (35/143)
   📈 Progress: 29% (42/143)
   📈 Progress: 34% (49/143)
   📈 Progress: 39% (56/143)
   📈 Progress: 44% (63/143)
   📈 Progress: 49% (70/143)
   📈 Progress: 54% (77/143)
   📈 Progress: 59% (84/

In [6]:
# === COMPARISON WITH CRUDE FILTERING ===

# Load model for comparison
rr = te.loadSBMLModel(MODEL_PATH)

# Compare with legacy approach
comparison = compare_with_crude_filtering(rr, active_params)

print(f"\n✅ Analysis Complete!")
print(f"   The rigorous approach provides a {comparison['reduction_factor']:.1f}x improvement")
print(f"   over crude name-based filtering.")


🆚 COMPARISON WITH CRUDE NAME-BASED FILTERING
📊 Comparison Results:
   Rigorous analysis: 127 parameters
   Crude filtering: 156 parameters
   Common parameters: 122
   Improvement: 1.2x fewer parameters

   Parameters incorrectly included by crude method (34):
     F
     Vd2c1
     Vicdh
     ec1b5
     mu
     ... and 29 more

   Parameters correctly identified by rigorous method (5):
     CDC15T
     ESP1T
     IET
     TEM1T
     kdirent

✅ Analysis Complete!
   The rigorous approach provides a 1.2x improvement
   over crude name-based filtering.


In [7]:
# === DISPLAY PARAMETER DETAILS ===

print("📋 DETAILED PARAMETER ANALYSIS")
print("="*50)

# Show top 20 most sensitive parameters
sorted_by_sensitivity = sorted(sensitivities.items(), key=lambda x: x[1], reverse=True)
top_sensitive = sorted_by_sensitivity[:20]

print(f"🔥 Top 20 Most Sensitive Parameters:")
for i, (param, sensitivity) in enumerate(top_sensitive, 1):
    value = active_params.get(param, all_free_params.get(param, 'N/A'))
    print(f"   {i:2d}. {param:<12} | sensitivity: {sensitivity:.2e} | value: {value}")

# Show least sensitive but still active parameters
active_sensitivities = [(p, s) for p, s in sensitivities.items() if p in active_params]
least_sensitive = sorted(active_sensitivities, key=lambda x: x[1])[:10]

print(f"\n🔍 Least Sensitive Active Parameters (but still above threshold):")
for i, (param, sensitivity) in enumerate(least_sensitive, 1):
    value = active_params[param]
    print(f"   {i:2d}. {param:<12} | sensitivity: {sensitivity:.2e} | value: {value}")

print(f"\n📊 Statistical Summary:")
active_sensitivities_values = [s for p, s in active_sensitivities]
print(f"   Mean sensitivity: {np.mean(active_sensitivities_values):.2e}")
print(f"   Median sensitivity: {np.median(active_sensitivities_values):.2e}")
print(f"   Max sensitivity: {np.max(active_sensitivities_values):.2e}")
print(f"   Min sensitivity: {np.min(active_sensitivities_values):.2e}")

📋 DETAILED PARAMETER ANALYSIS
🔥 Top 20 Most Sensitive Parameters:
    1. ksspn        | sensitivity: 1.39e+00 | value: 0.1
    2. mdt          | sensitivity: 1.17e+00 | value: 90.0
    3. kdspn        | sensitivity: 7.60e-01 | value: 0.06
    4. ksb2_p_p     | sensitivity: 7.36e-01 | value: 0.04
    5. Jspn         | sensitivity: 3.40e-01 | value: 0.14
    6. kd20         | sensitivity: 3.12e-01 | value: 0.3
    7. ks20_p_p     | sensitivity: 2.23e-01 | value: 0.6
    8. kaiep        | sensitivity: 1.84e-01 | value: 0.1
    9. kdb2p        | sensitivity: 1.57e-01 | value: 0.15
   10. kiiep        | sensitivity: 1.56e-01 | value: 0.15
   11. kdnet        | sensitivity: 1.47e-01 | value: 0.03
   12. ksnet        | sensitivity: 1.38e-01 | value: 0.084
   13. ks14         | sensitivity: 1.32e-01 | value: 0.2
   14. kimcm        | sensitivity: 1.19e-01 | value: 0.15
   15. kd14         | sensitivity: 1.18e-01 | value: 0.1
   16. kamcm        | sensitivity: 1.09e-01 | value: 1.0
   17. ka20_

In [12]:
def get_structurally_free_parameters(model_path: str) -> Dict[str, float]:
    """
    Returns parameters that are mathematically free to vary.
    No sensitivity testing - just structural analysis.
    """
    sbml_doc = libsbml.readSBML(model_path)
    model = sbml_doc.getModel()
    
    # Start with all parameters
    all_params = {p.getId(): p.getValue() 
                  for p in model.getListOfParameters()}
    
    # Remove parameters defined by assignment rules
    rule_params = set()
    for rule in model.getListOfRules():
        if isinstance(rule, (libsbml.AssignmentRule, libsbml.RateRule)):
            rule_params.add(rule.getVariable())
    
    # Remove parameters with initial assignments
    init_params = set()
    for init in model.getListOfInitialAssignments():
        init_params.add(init.getSymbol())
    
    # Keep only free parameters
    free_params = {k: v for k, v in all_params.items() 
                   if k not in rule_params and k not in init_params}
    
    return free_params


# Just use this:
free_parameters = get_structurally_free_parameters(MODEL_PATH)

print(f"Structurally free parameters: {len(free_parameters)}")
print("These are ALL parameters you CAN vary (not constrained by rules)")

# Use this list in your ChicoOscillation code
RIGOROUSLY_FREE_PARAMETERS = list(free_parameters.keys())

Structurally free parameters: 143
These are ALL parameters you CAN vary (not constrained by rules)


In [6]:
# === SLOPPYCELL COMPARISON ===
# Try to import SloppyCell at module level
try:
    import SloppyCell
    import SloppyCell.ExampleNets as ExampleNets
    SLOPPYCELL_AVAILABLE = True
except ImportError:
    SLOPPYCELL_AVAILABLE = False

def compare_with_sloppycell():
    """
    Compare our structural analysis with SloppyCell's parameter list.
    SloppyCell has a built-in Chen 2004 model.
    """
    print("\n" + "="*70)
    print("COMPARING WITH SLOPPYCELL")
    print("="*70)
    
    if not SLOPPYCELL_AVAILABLE:
        print("\n❌ SloppyCell is not installed")
        print("\n📥 To install SloppyCell in your sloppycell venv:")
        print("   mamba activate sloppycell")
        print("   mamba install -c conda-forge sloppycell")
        print("   or if not available in conda-forge:")
        print("   pip install SloppyCell")
        return None
    
    print("\n✓ SloppyCell is installed")
    
    # Load SloppyCell's Chen model
    try:
        # Check if Chen model is available
        if hasattr(ExampleNets, 'Chen2004'):
            chen_net = ExampleNets.Chen2004.net
            
            # Get SloppyCell's parameter list
            sloppy_params = chen_net.get_param_names()
            
            print("\n📊 SloppyCell Chen2004 model:")
            print("   Parameters: {}".format(len(sloppy_params)))
            print("\n   First 20 parameters:")
            for i, p in enumerate(sloppy_params[:20], 1):
                print("   {:2d}. {}".format(i, p))
            
            return sloppy_params
        else:
            print("\n⚠ Chen2004 model not found in SloppyCell.ExampleNets")
            print("   Available models:", dir(ExampleNets))
            
    except Exception as e:
        print("\n⚠ Could not load SloppyCell's Chen model: {}".format(e))
        
    return None


# Try to get SloppyCell's parameter list
sloppy_params = compare_with_sloppycell()

if sloppy_params:
    # Compare with our structural analysis
    print("\n" + "="*70)
    print("PARAMETER SET COMPARISON")
    print("="*70)
    
    our_params = set(free_parameters.keys())
    sloppy_set = set(sloppy_params)
    
    print("\n   Your SBML model:     {} parameters".format(len(our_params)))
    print("   SloppyCell model:    {} parameters".format(len(sloppy_set)))
    
    # Find differences
    only_in_ours = our_params - sloppy_set
    only_in_sloppy = sloppy_set - our_params
    common = our_params & sloppy_set
    
    print("\n   Common parameters:   {}".format(len(common)))
    print("   Only in your model:  {}".format(len(only_in_ours)))
    print("   Only in SloppyCell:  {}".format(len(only_in_sloppy)))
    
    if only_in_ours:
        print("\n   Parameters only in your SBML model:")
        for p in sorted(only_in_ours)[:20]:
            print("      • {}".format(p))
        if len(only_in_ours) > 20:
            print("      ... and {} more".format(len(only_in_ours) - 20))
    
    if only_in_sloppy:
        print("\n   Parameters only in SloppyCell:")
        for p in sorted(only_in_sloppy)[:20]:
            print("      • {}".format(p))
        if len(only_in_sloppy) > 20:
            print("      ... and {} more".format(len(only_in_sloppy) - 20))


COMPARING WITH SLOPPYCELL

❌ SloppyCell is not installed

📥 To install SloppyCell in your sloppycell venv:
   mamba activate sloppycell
   mamba install -c conda-forge sloppycell
   or if not available in conda-forge:
   pip install SloppyCell
